1.   Cài đặt môi trường

In [ ]:
%pip install pandas numpy scikit-learn xgboost imbalanced-learn joblib

2.   Import thư viện


In [12]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.metrics import classification_report, roc_auc_score, precision_score, recall_score, f1_score

# Models
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier

import joblib

3. Load UCI dataset

In [13]:
columns = [
    "status", "duration", "credit_history", "purpose", "credit_amount",
    "savings", "employment", "installment_rate", "personal_status",
    "other_debtors", "residence_since", "property", "age",
    "other_installment_plans", "housing", "existing_credits",
    "job", "people_liable", "telephone", "foreign_worker", "target"
]

df = pd.read_csv("german.data", sep=" ", header=None, names=columns)

# fix label
df["target"] = df["target"].map({1: 1, 2: 0})

df.head()

4. Slipt data

In [14]:
X = df.drop("target", axis=1)
y = df["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

5. Preprocessing

In [15]:
num_cols = X.select_dtypes(include=['int64','float64']).columns
cat_cols = X.select_dtypes(include=['object']).columns

preprocessor = ColumnTransformer([
    ("num", StandardScaler(), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols)
])

6. Define models

In [16]:
models = {
    "Logistic": LogisticRegression(class_weight={1:1, 0:5}, max_iter=1000),
    "SVM": SVC(
        kernel="rbf",
        probability=True,
        class_weight={1:1, 0:5}
    ),
    "DecisionTree": DecisionTreeClassifier(),
    "RandomForest": RandomForestClassifier(n_estimators=200,class_weight={1:1, 0:5}),
    "GradientBoosting": GradientBoostingClassifier(),
    "XGBoost": XGBClassifier(
        eval_metric="logloss",
        use_label_encoder=False
    )
}

7. Train tất cả model

In [17]:
results = []
best_model = None
best_score = 0

for name, model in models.items():
    print("="*50)
    print(f"Training {name}")

    pipeline = Pipeline([
        ("preprocess", preprocessor),
        ("model", model)
    ])

    pipeline.fit(X_train, y_train)

    y_pred = pipeline.predict(X_test)
    y_prob = pipeline.predict_proba(X_test)[:,1]

    roc = roc_auc_score(y_test, y_prob)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)

    print(classification_report(y_test, y_pred))
    print("ROC-AUC:", roc)

    results.append({
        "model": name,
        "roc_auc": roc,
        "precision": precision,
        "recall": recall,
        "f1": f1
    })

    if roc > best_score:
        best_score = roc
        best_model = pipeline

8. So sánh kết quả

In [18]:
df_results = pd.DataFrame(results)
df_results = df_results.sort_values(by="roc_auc", ascending=False)

df_results

9. Biểu diễn kết quả

In [19]:
import matplotlib.pyplot as plt

df_results.set_index("model")["roc_auc"].plot(kind="bar")
plt.title("Model Comparison (ROC-AUC)")
plt.show()

10. Lưu kết quả

In [20]:
print("Best model:", df_results.iloc[0]["model"])

joblib.dump(best_model, "best_model.pkl")

## 11. Học liên tục (Continuous Learning) - Huấn luyện lại mô hình tốt nhất với dữ liệu mới

Để mô phỏng quá trình học liên tục, chúng ta sẽ giả định có một tập dữ liệu mới (ví dụ, một phần nhỏ của `X_train` ban đầu) và huấn luyện lại mô hình `best_model`.

In [ ]:
# Giả lập dữ liệu mới (ví dụ: lấy một phần nhỏ từ tập huấn luyện ban đầu)
X_new, _, y_new, _ = train_test_split(X_train, y_train, test_size=0.8, random_state=1, stratify=y_train)

print(f"Số lượng mẫu dữ liệu mới: {len(X_new)}")
display(X_new.head())

In [ ]:
# Huấn luyện lại mô hình tốt nhất với dữ liệu mới (hoặc kết hợp với dữ liệu cũ)
# Trong thực tế, bạn có thể gộp X_train và X_new lại hoặc chỉ dùng X_new tùy chiến lược.
# Ở đây, chúng ta sẽ huấn luyện lại trên toàn bộ X_train + X_new để minh họa cập nhật.

# Kết hợp dữ liệu cũ và dữ liệu mới (đảm bảo cột tương thích)
X_combined = pd.concat([X_train, X_new], ignore_index=True)
y_combined = pd.concat([y_train, y_new], ignore_index=True)

print("Bắt đầu huấn luyện lại mô hình tốt nhất với dữ liệu kết hợp...")
best_model.fit(X_combined, y_combined)
print("Huấn luyện lại hoàn tất!")

# Đánh giá lại mô hình sau khi huấn luyện lại
y_pred_retrained = best_model.predict(X_test)
y_prob_retrained = best_model.predict_proba(X_test)[:,1]

roc_retrained = roc_auc_score(y_test, y_prob_retrained)
precision_retrained = precision_score(y_test, y_pred_retrained)
recall_retrained = recall_score(y_test, y_pred_retrained)
f1_retrained = f1_score(y_test, y_pred_retrained)

print("="*50)
print("Kết quả sau khi huấn luyện lại:")
print(classification_report(y_test, y_pred_retrained))
print(f"ROC-AUC (sau huấn luyện lại): {roc_retrained}")
print("="*50)

Cách tiếp cận trên giúp bạn cập nhật mô hình với dữ liệu mới. Trong một hệ thống sản xuất, quá trình này có thể được tự động hóa để chạy định kỳ hoặc khi có đủ lượng dữ liệu mới.

11. **Turning**

In [21]:
y_prob = best_model.predict_proba(X_test)[:,1]
y_pred = (y_prob > 0.4).astype(int)

print(classification_report(y_test, y_pred))

In [22]:
from sklearn.metrics import confusion_matrix
print(confusion_matrix(y_test, y_pred))

In [23]:
model = RandomForestClassifier()
pipeline = Pipeline([
    ("preprocess", preprocessor),
    ("model", model)
])

pipeline.fit(X_train, y_train)

importances = model.feature_importances_

In [24]:
feature_names = pipeline.named_steps["preprocess"].get_feature_names_out()

In [25]:
import pandas as pd

feat_imp = pd.DataFrame({
    "feature": feature_names,
    "importance": importances
}).sort_values(by="importance", ascending=False)

feat_imp.head(10)